# 📊 SentinelIQ — Notebook 03: Model Training & Evaluation

This notebook trains and evaluates all three SentinelIQ model architectures on the NASA C-MAPSS dataset:

| Model | Architecture | Target RMSE |
|-------|-------------|-------------|
| LSTM | Bidirectional 2-layer LSTM | ≤ 18 cycles |
| TCN  | Temporal Convolutional Network | ≤ 13 cycles |
| MultiTask TCN | Shared TCN + RUL/failure heads | ≤ 16 cycles |

**Evaluation metrics:** RMSE · MAE · R² · NASA Asymmetric Score

In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────────
import sys, os
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
from torch.utils.data import DataLoader

from src.config import config
from src.data_loader import CMAPSSLoader
from src.features import FeatureEngineer
from src.models import (
    CMAPSSDataset, CMAPSSDataset_MultiTask,
    LSTMModel, TCNModel, MultiTaskTCN, Autoencoder
)
from src.trainer import ModelTrainer
from src.evaluator import ModelEvaluator

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
COLORS = sns.color_palette('husl', 8)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Imports loaded")
print(f"   PyTorch: {torch.__version__}")
print(f"   Device : {DEVICE}")

In [ ]:
# ── Data loading + feature engineering ────────────────────────────────────────
DATA_DIR = REPO_ROOT / 'data' / 'raw'
loader = CMAPSSLoader(data_dir=DATA_DIR)

try:
    train_raw, test_raw, rul_df = loader.load_single_dataset('FD001')
    print(f"✅ Real FD001 loaded")
except FileNotFoundError:
    print("⚠️  Real data not found. Using synthetic data for demonstration...")
    np.random.seed(42)
    cols = ['unit_id','cycle','op_setting_1','op_setting_2','op_setting_3'] + \
           [f'sensor_{i}' for i in range(1, 22)]
    rows = []
    for uid in range(1, 101):
        life = np.random.randint(80, 200)
        for c in range(1, life + 1):
            degradation = c / life
            sensors = np.random.randn(21) * 0.5 + degradation * np.array(
                [1,-1,0.5,-0.5,2,-2,0.8,-0.8,1.2,-1.2,0,0.3,-0.3,0.7,-0.7,0,0,0,0.4,-0.4,0.6]
            )
            for ci in [0,4,9,15,17,18]: sensors[ci] = 0.0
            row = [uid, c, 0.0, 0.0002, 100.0] + sensors.tolist()
            rows.append(row)
    train_raw = pd.DataFrame(rows, columns=cols).astype({'unit_id':'int32','cycle':'int32'})
    test_rows = []
    for uid in range(1, 101):
        life = np.random.randint(40, 100)
        for c in range(1, life+1):
            sensors = np.random.randn(21) * 0.5
            for ci in [0,4,9,15,17,18]: sensors[ci] = 0.0
            test_rows.append([uid, c, 0.0, 0.0002, 100.0] + sensors.tolist())
    test_raw = pd.DataFrame(test_rows, columns=cols).astype({'unit_id':'int32','cycle':'int32'})
    rul_df = pd.DataFrame({'RUL': np.random.randint(0, 80, size=100)})

# Feature engineering
fe = FeatureEngineer(
    rul_cap=config.rul_cap,
    variance_threshold=config.variance_threshold,
    n_operating_clusters=config.n_operating_clusters,
    random_seed=config.random_seed,
)
train_proc, test_proc, feature_cols = fe.process(train_raw.copy(), test_raw.copy())
print(f"Feature cols: {len(feature_cols)} → {feature_cols}")

## 1. Model Architecture Comparison

In [ ]:
# ── Model architecture summary ─────────────────────────────────────────────────
INPUT_DIM = len(feature_cols)
SEQ_LEN   = config.sequence_length

lstm_model  = LSTMModel(input_dim=INPUT_DIM, hidden_dim=config.hidden_dim, dropout=config.dropout_rate)
tcn_model   = TCNModel(input_dim=INPUT_DIM, num_channels=config.tcn_channels, kernel_size=config.tcn_kernel_size, dropout=config.dropout_rate)
mt_model    = MultiTaskTCN(input_dim=INPUT_DIM, num_channels=config.tcn_channels, n_failure_modes=config.n_failure_modes, dropout=config.dropout_rate)
ae_model    = Autoencoder(input_dim=INPUT_DIM)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

arch_data = {
    'Model': ['LSTM (Bidirectional)', 'TCN', 'MultiTask TCN', 'Autoencoder'],
    'Parameters': [count_params(m) for m in [lstm_model, tcn_model, mt_model, ae_model]],
    'Input Shape': [f'(B, {SEQ_LEN}, {INPUT_DIM})'] * 3 + [f'(B, {INPUT_DIM})'],
    'Output': ['(B,) RUL', '(B,) RUL', '(B,) RUL + (B,5) modes', f'(B, {INPUT_DIM}) reconstruction'],
    'Target RMSE': ['≤ 18 cycles', '≤ 13 cycles', '≤ 16 cycles', 'Anomaly detection'],
}
arch_df = pd.DataFrame(arch_data)
arch_df['Parameters'] = arch_df['Parameters'].apply(lambda x: f'{x:,}')
display(arch_df.set_index('Model'))

## 2. Dataset Preparation

In [ ]:
# ── Build DataLoaders ──────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

# Split training units into train/val (80/20 by unit_id)
all_units = train_proc['unit_id'].unique()
train_units, val_units = train_test_split(all_units, test_size=0.2, random_state=42)

train_subset = train_proc[train_proc['unit_id'].isin(train_units)]
val_subset   = train_proc[train_proc['unit_id'].isin(val_units)]

# Add synthetic failure mode labels for MultiTask demo
np.random.seed(42)
for uid in train_proc['unit_id'].unique():
    mode = np.random.randint(0, config.n_failure_modes)
    train_proc.loc[train_proc['unit_id'] == uid, 'failure_mode'] = mode
train_proc['failure_mode'] = train_proc['failure_mode'].astype(int)

BATCH_SIZE = config.batch_size
SEQ_LEN    = config.sequence_length

train_ds    = CMAPSSDataset(train_proc[train_proc['unit_id'].isin(train_units)], feature_cols, SEQ_LEN)
val_ds      = CMAPSSDataset(train_proc[train_proc['unit_id'].isin(val_units)],   feature_cols, SEQ_LEN)
train_mt_ds = CMAPSSDataset_MultiTask(train_proc[train_proc['unit_id'].isin(train_units)], feature_cols, SEQ_LEN)
val_mt_ds   = CMAPSSDataset_MultiTask(train_proc[train_proc['unit_id'].isin(val_units)],   feature_cols, SEQ_LEN)

train_loader    = DataLoader(train_ds,    batch_size=BATCH_SIZE, shuffle=True)
val_loader      = DataLoader(val_ds,      batch_size=BATCH_SIZE, shuffle=False)
train_mt_loader = DataLoader(train_mt_ds, batch_size=BATCH_SIZE, shuffle=True)
val_mt_loader   = DataLoader(val_mt_ds,   batch_size=BATCH_SIZE, shuffle=False)

print(f"Train sequences : {len(train_ds):,}")
print(f"Val sequences   : {len(val_ds):,}")
print(f"Sequence shape  : {train_ds[0][0].shape}  (seq_len × features)")

## 3. Train Models

> ⚡ **For quick demo:** `config.n_epochs` is used. Reduce to 5–10 epochs for fast notebook execution.
> Full training (50 epochs) typically takes ~5 min on CPU, ~45s on GPU.

In [ ]:
# ── Training with short epoch count for notebook demo ─────────────────────────
import copy, time
from dataclasses import dataclass, field
from typing import List

# Override config for demo (5 epochs instead of 50)
demo_config = copy.copy(config)
demo_config.n_epochs  = 5
demo_config.patience  = 5

trainer = ModelTrainer(demo_config)
INPUT_DIM = len(feature_cols)
MODEL_DIR = REPO_ROOT / 'models'

print("Training LSTM (5 epochs demo)...")
t0 = time.time()
trained_lstm = trainer.train_lstm(
    train_loader, val_loader, input_dim=INPUT_DIM,
    save_path=MODEL_DIR / 'demo_lstm.pth'
)
print(f"LSTM done in {time.time()-t0:.1f}s")

print("\nTraining TCN (5 epochs demo)...")
t0 = time.time()
trained_tcn = trainer.train_tcn(
    train_loader, val_loader, input_dim=INPUT_DIM,
    save_path=MODEL_DIR / 'demo_tcn.pth'
)
print(f"TCN done in {time.time()-t0:.1f}s")

## 4. Evaluation Metrics

In [ ]:
# ── Evaluate on validation set ─────────────────────────────────────────────────
evaluator = ModelEvaluator()

# Ground truth from val DataLoader
val_true = np.array([t.item() for _, t in val_ds])

print("Evaluating LSTM...")
lstm_metrics = evaluator.evaluate(trained_lstm, val_loader, val_true, model_name='LSTM')

print("\nEvaluating TCN...")
tcn_metrics  = evaluator.evaluate(trained_tcn, val_loader, val_true, model_name='TCN')

# Comparison
all_results = {'LSTM': lstm_metrics, 'TCN': tcn_metrics}
best_model  = evaluator.compare_models(all_results)
print(f"\n🏆 Best model: {best_model}")

In [ ]:
# ── Metrics comparison bar chart ───────────────────────────────────────────────
metrics_names = ['RMSE', 'MAE', 'R2']
model_names   = list(all_results.keys())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric in zip(axes, metrics_names):
    values = [all_results[m][metric] for m in model_names]
    bars   = ax.bar(model_names, values, color=[COLORS[i] for i in range(len(model_names))],
                    edgecolor='white', width=0.5)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel('Value')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01*max(values),
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.suptitle('Model Performance Comparison (5-epoch demo)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'outputs' / 'eval_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Predicted vs Actual RUL

In [ ]:
# ── Predicted vs actual scatter ────────────────────────────────────────────────
def get_predictions(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for seq, _ in loader:
            preds.append(model(seq.to(device)).cpu().numpy())
    return np.concatenate(preds)

lstm_preds = get_predictions(trained_lstm, val_loader, DEVICE)
tcn_preds  = get_predictions(trained_tcn,  val_loader, DEVICE)

# Clip negative predictions to 0
lstm_preds = np.clip(lstm_preds, 0, None)
tcn_preds  = np.clip(tcn_preds,  0, None)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
max_val = max(val_true.max(), lstm_preds.max(), tcn_preds.max())

for ax, preds, name, color in zip(axes,
                                   [lstm_preds, tcn_preds],
                                   ['LSTM', 'TCN'],
                                   [COLORS[0], COLORS[2]]):
    ax.scatter(val_true, preds, alpha=0.3, s=10, color=color)
    ax.plot([0, max_val], [0, max_val], 'k--', lw=1.5, label='Perfect')
    ax.set_xlabel('True RUL (cycles)')
    ax.set_ylabel('Predicted RUL (cycles)')
    ax.set_title(f'{name} — Predicted vs Actual', fontweight='bold')
    rmse = np.sqrt(np.mean((preds - val_true)**2))
    ax.text(0.05, 0.95, f'RMSE={rmse:.1f} cycles', transform=ax.transAxes,
            va='top', fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
    ax.legend()

plt.suptitle('Predicted vs Actual RUL (Validation Set)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'outputs' / 'eval_pred_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Residual Analysis

In [ ]:
# ── Residual plots ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, (preds, name, color) in enumerate([
    (lstm_preds, 'LSTM', COLORS[0]),
    (tcn_preds,  'TCN',  COLORS[2])
]):
    residuals = preds - val_true

    # Residuals vs predicted
    axes[0, i].scatter(preds, residuals, alpha=0.3, s=8, color=color)
    axes[0, i].axhline(0, color='crimson', lw=2, linestyle='--')
    axes[0, i].set_xlabel('Predicted RUL')
    axes[0, i].set_ylabel('Residual (Pred − True)')
    axes[0, i].set_title(f'{name} — Residuals vs Predicted', fontweight='bold')

    # Residual histogram
    axes[1, i].hist(residuals, bins=40, color=color, edgecolor='white', alpha=0.85)
    axes[1, i].axvline(0, color='crimson', lw=2, linestyle='--')
    axes[1, i].axvline(residuals.mean(), color='navy', lw=1.5, linestyle=':',
                       label=f'Mean={residuals.mean():.1f}')
    axes[1, i].set_xlabel('Residual (cycles)')
    axes[1, i].set_ylabel('Count')
    axes[1, i].set_title(f'{name} — Residual Distribution', fontweight='bold')
    axes[1, i].legend()

plt.tight_layout()
plt.savefig(REPO_ROOT / 'outputs' / 'eval_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Anomaly Detector Demo

In [ ]:
# ── Autoencoder anomaly detection demo ────────────────────────────────────────
from src.anomaly import AnomalyDetector

# Fit anomaly detector on training data
detector = AnomalyDetector(
    contamination=config.anomaly_contamination,
    healthy_rul_threshold=config.anomaly_healthy_rul_threshold,
    critical_threshold=config.anomaly_critical_threshold,
    warning_threshold=config.anomaly_warning_threshold,
)

# Use processed training data
detector.fit(train_proc, feature_cols)
anomaly_results = detector.predict(test_proc, feature_cols)

# Score distribution
scores = anomaly_results['anomaly_score'].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(scores, bins=40, color=COLORS[3], edgecolor='white', alpha=0.85)
axes[0].axvline(config.anomaly_warning_threshold, color='orange', lw=2,
                linestyle='--', label=f'Warning ({config.anomaly_warning_threshold})')
axes[0].axvline(config.anomaly_critical_threshold, color='crimson', lw=2,
                linestyle='--', label=f'Critical ({config.anomaly_critical_threshold})')
axes[0].set_xlabel('Anomaly Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Anomaly Score Distribution', fontweight='bold')
axes[0].legend()

# Severity breakdown
severity_counts = anomaly_results['severity'].value_counts()
palette = {'normal': '#2ecc71', 'warning': '#f39c12', 'critical': '#e74c3c'}
bar_colors = [palette.get(s, '#95a5a6') for s in severity_counts.index]
axes[1].bar(severity_counts.index, severity_counts.values, color=bar_colors, edgecolor='white')
axes[1].set_title('Anomaly Severity Breakdown', fontweight='bold')
axes[1].set_ylabel('Number of Time Steps')
for i, (sev, cnt) in enumerate(severity_counts.items()):
    pct = 100 * cnt / len(anomaly_results)
    axes[1].text(i, cnt + 5, f'{pct:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(REPO_ROOT / 'outputs' / 'eval_anomaly_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n📊 Anomaly detection on {len(anomaly_results):,} test datapoints")
print(severity_counts.to_string())

## 8. Final Results Summary

| Metric | LSTM | TCN | Target |
|--------|------|-----|--------|
| RMSE (cycles) | *(see above)* | *(see above)* | TCN ≤ 13, LSTM ≤ 18 |
| MAE (cycles)  | — | — | — |
| R² | — | — | ≥ 0.90 |
| NASA Score | — | — | Lower is better |

> **Note:** Results above are from a 5-epoch demo. For production-quality 
> results, run `pipeline/sentineliq_ml_pipeline.py` with `n_epochs=50`.

---

## Next Steps

1. **Phase 15** — FastAPI backend: `ml_server/main.py`
2. **Phase 16** — PostgreSQL schema + Supabase integration
3. **Phase 17** — Docker containerization
4. **Phase 18–21** — Next.js dashboard frontend